In [ ]:
#from google.colab import drive; drive.mount('/content/drive')
import os 
print(os.listdir('/content/drive/MyDrive/data/names'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Russian.txt', 'Korean.txt', 'Portuguese.txt', 'Chinese.txt', 'Czech.txt', 'Irish.txt', 'Scottish.txt', 'Arabic.txt', 'Italian.txt', 'Polish.txt', 'English.txt', 'German.txt', 'Spanish.txt', 'Dutch.txt', 'Greek.txt', 'French.txt', 'Japanese.txt', 'Vietnamese.txt']


In [20]:
import torch
import string 
import unicodedata

In [21]:

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda") 
torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")


Using device = cpu


In [22]:
allowed_chars = string.ascii_letters + " .,;'" + "_"
n_lets = len(allowed_chars) 

def unicodeToAscii(s): 
    return ''.join(
        c for c in unicodedata.normalize('NFD', s) 
        if unicodedata.category(c) != 'Mn'
        and c if allowed_chars
    )

###
print(f"Converting 'Slusarski' to {unicodeToAscii('Slusarski')}")
###

Converting 'Slusarski' to Slusarski


In [23]:
def letter_to_indx(letter): 
    if letter not in allowed_chars:
        return allowed_chars.find("_") 
    else: 
        return allowed_chars.find(letter) 

def lineToTensor(line): 
    tensor = torch.zeros(len(line), 1, n_lets)
    for li, letter in enumerate(line): 
        tensor[li][0][letter_to_indx(letter)] = 1
    return tensor 


print(f"The letter 'a' becomes{lineToTensor('a')}")
print(f"The name 'Ahn' becomes {lineToTensor('Ahn')}")


The letter 'a' becomestensor([[[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]]])
The name 'Ahn' becomes tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.

In [27]:

import glob 
import os 
import time 
import torch 
from torch.utils.data import Dataset 
from io import open 

class NamesDataset(Dataset):

    def __init__(self, data_dir): 
        self.data_dir = data_dir
        self.load_time = time.localtime
        labels_set = set() 

        self.data = []
        self.data_tensors = [] 
        self.labels = [] 
        self.labels_tensors = [] 

        text_files = glob.glob(os.path.join(data_dir, '*.txt')) 
        for filename in text_files: 
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label) 
            lines = open(filename, encoding='utf-8').read().strip().split('\n')
            for name in lines: 
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label) 

        self.labels_uniq = list(labels_set)
        for idx in range(len(self.labels)): 
            tmp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long) 
            self.labels_tensors.append(tmp_tensor) 


    def __len__(self): 
        return len(self.data) 

    def __getitem__(self, idx):
        data_item = self.data[idx] 
        data_label = self.labels[idx]
        data_tensor = self.data_tensors[idx] 
        label_tensor = self.labels_tensors[idx] 

        return label_tensor, data_tensor, data_label, data_item

    

In [25]:
print(os.path.abspath("rneuralnets/data/names"))
print(glob.glob(os.path.join("/content/drive/MyDrive/data/names", "*.txt")))

/content/rneuralnets/data/names
['/content/drive/MyDrive/data/names/Russian.txt', '/content/drive/MyDrive/data/names/Korean.txt', '/content/drive/MyDrive/data/names/Portuguese.txt', '/content/drive/MyDrive/data/names/Chinese.txt', '/content/drive/MyDrive/data/names/Czech.txt', '/content/drive/MyDrive/data/names/Irish.txt', '/content/drive/MyDrive/data/names/Scottish.txt', '/content/drive/MyDrive/data/names/Arabic.txt', '/content/drive/MyDrive/data/names/Italian.txt', '/content/drive/MyDrive/data/names/Polish.txt', '/content/drive/MyDrive/data/names/English.txt', '/content/drive/MyDrive/data/names/German.txt', '/content/drive/MyDrive/data/names/Spanish.txt', '/content/drive/MyDrive/data/names/Dutch.txt', '/content/drive/MyDrive/data/names/Greek.txt', '/content/drive/MyDrive/data/names/French.txt', '/content/drive/MyDrive/data/names/Japanese.txt', '/content/drive/MyDrive/data/names/Vietnamese.txt']


In [28]:
alldata = NamesDataset("/content/drive/MyDrive/data/names")
print(f"loaded {len(alldata)} items of data")
print(f"example = {alldata[0]}")


loaded 20074 items of data
example = (tensor([14]), tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [ ]:
# print(len(alldata.data), len(alldata.labels), len(alldata.data_tensors))
# print(len(alldata))

In [30]:
train_set, test_set = torch.utils.data.random_split(alldata, [.85, .15], generator=torch.Generator(device=device).manual_seed(2024))
print(f"train exampels = {len(train_set)}, validation examples = {len(test_set)}")


train exampels = 17063, validation examples = 3011


In [33]:
import torch.nn as nn 
import torch.nn.functional as F

class CharRNN(nn.Module):
    def __init__(self, inp_size, hid_size, outp_size): 
        super(CharRNN, self).__init__() 

        self.rnn = nn.RNN(inp_size, hid_size)
        self.h2o = nn.Linear(hid_size, outp_size) 
        self.softmax = nn.LogSoftmax(dim=1) 

    def forward(self, line_tensr): 
        rnn_out, hidden  = self.rnn(line_tensr)
        outp = self.h2o(hidden[0]) 
        outp = self.softmax(outp)

        return outp

n_hidden = 128 
rnn = CharRNN(n_lets, n_hidden, len(alldata.labels_uniq))
print(rnn) 




CharRNN(
  (rnn): RNN(58, 128)
  (h2o): Linear(in_features=128, out_features=18, bias=True)
  (softmax): LogSoftmax(dim=1)
)


In [34]:
def label_from_output(output, output_labels): 
    top_n, top_i = output.topk(1) 
    label_i = top_i[0].item() 
    return output_labels[label_i], label_i

input = lineToTensor('Albert') 
output = rnn(input)
print(output) 
print(label_from_output, alldata.labels_uniq)

tensor([[-2.9177, -3.0320, -2.9156, -2.9831, -2.9905, -2.8004, -2.7101, -2.9946,
         -3.0420, -2.8475, -2.9543, -2.9416, -2.8058, -2.9212, -2.8031, -2.9723,
         -2.6976, -2.7944]], grad_fn=<LogSoftmaxBackward0>)
<function label_from_output at 0x7d2cca58a700> ['Chinese', 'Scottish', 'Czech', 'Italian', 'Polish', 'Spanish', 'French', 'Greek', 'Portuguese', 'Arabic', 'Japanese', 'Irish', 'German', 'Vietnamese', 'Russian', 'Korean', 'Dutch', 'English']


In [39]:
import random 
import numpy as np 


def train(rnn,  training_data, n_epoch=10, n_batch_size=64, report=50, learn_rate=0.2, criterion=nn.NLLLoss()):
    current_loss = 0 
    all_losses = [] 
    rnn.train()
    optimizer = torch.optim.SGD(rnn.parameters(), lr=learn_rate)

    start = time.time() 

    for iter in range(1, n_epoch + 1): 
        rnn.zero_grad() 

        batches = list(range(len(training_data))) 
        random.shuffle(batches) 
        batches = np.array_split(batches, len(batches) // n_batch_size)

        for idx, batch in enumerate(batches): 
            batch_loss = 0 

            for i in batch: 
                (label_tensor, txt_tensor, label, txt) = training_data[i]
                output = rnn.forward(txt_tensor)
                loss = criterion(output, label_tensor)
                batch_loss += loss

            batch_loss.backward()
            nn.utils.clip_grad_norm_(rnn.parameters(), 3)
            optimizer.step() 
            optimizer.zero_grad() 

            current_loss += batch_loss.item() / len(batch)
        all_losses.append(current_loss / len(batches)) 
        if iter % report == 0:
            print(f"{iter} ({iter / n_epoch:.0%}): \t average batch loss = {all_losses[-1]}")
        current_loss = 0 

    return all_losses

In [40]:
start = time.time() 
all_losses = train(rnn, train_set, n_epoch=30, learn_rate=0.2, report=3)
end = time.time() 
print(f"training took {end-start}s")


3 (10%): 	 average batch loss = 1.0169500515631955
6 (20%): 	 average batch loss = 0.8513303891332802
9 (30%): 	 average batch loss = 0.7444869674134627
12 (40%): 	 average batch loss = 0.6738516750950699
15 (50%): 	 average batch loss = 0.6231723790812727
18 (60%): 	 average batch loss = 0.581516571792824
21 (70%): 	 average batch loss = 0.5521739624962081
24 (80%): 	 average batch loss = 0.5331586142042042
27 (90%): 	 average batch loss = 0.5207193540115105
30 (100%): 	 average batch loss = 0.5069436500015944
training took 1145.2045619487762s
